# Exercise 3: Strings, Functions, If Else, For Loops, Git

In [1]:
import altair as alt
import numpy as np
import pandas as pd
from calitp_data_analysis import calitp_color_palette

In [2]:
pd.options.display.max_columns = 100
pd.options.display.float_format = "{:.2f}".format
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

* Using a `f-string`, load in your merged dataframe from Exercise 3.

In [3]:
GCS_FILE_PATH = "gs://calitp-analytics-data/data-analyses/starter_kit/"
PROJECT_SCORES_BY_DISTRICT_NAME = "project_scores_by_district.parquet"  

In [4]:
project_scores_by_district_path = f"{GCS_FILE_PATH}{PROJECT_SCORES_BY_DISTRICT_NAME}"

In [5]:
project_scores = pd.read_parquet(project_scores_by_district_path)

## Categorizing
* There are 40+ projects. They all vary in themes, some contain transit elements while others contain Active Transportation (ATP) components. Some contain both! 
* Categorizing data is an important part of data cleaning and analyzing so we can present the data on a more succinct level. 
* Let's organize projects into three categories.
    * ATP
    * Transit
    * General Lanes

### Task 1: Strings
* Below are some of the common keywords that fall into the categories detailed above. They are held in a `list`.
* Add other terms you think are relevant. 
* We are going to search the `Scope of Work` column for these keywords. 

In [6]:
transit = ["transit", "passenger rail", "bus", "ferry", "brt"]
atp = ["bike", "pedestrian", "bicycle", "sidewalk", "path", "multiuse"]
general_lanes = ["general", "auxiliary", "highway", "hov", "hot", "managed lane"]

#### Step 1: Cleaning
* Remember in Exercise 2 some of the project names didn't merge between the two dataframes?
* In the real world, you won't have the bandwidth and time to replace each individual string value with a dictionary.
* An easy way to clean most of the values up is by lowercasing, stripping the white spaces, and replacing characters.
* We can search through a string column  easier when we simplify up the  values.

In [7]:
project_scores.scope_of_work = (
    project_scores.scope_of_work.str.lower() # Lowers the strings
    .str.strip() # Strips trailing white spaces
    .str.replace("-", " ") # Replaces hyphens with a space
    .str.replace(r"\+", " ", regex=True)
    .str.replace("_", " ")
)

* `str.contains()` allows you to search through the column. 
* Let's search for projects that have "transit" in their descriptions. 
* There are many modifications you can make to `str.contains()`. Try them out and see what happens.
    * `df.loc[df.scope_of_work.str.contains("transit", case=False)]` 
        * Will search through your column without matching the case. It'll return rows with both "Transit" and "transit".
    * `df.scope_of_work.str.contains("transit", case=False, regex=False) `
        * Will return any matches that include `transit` rather than an exact match. It'll return rows with values like "transit" and "Transitory".

In [8]:
project_scores_only_transit = project_scores.loc[project_scores.scope_of_work.str.contains("transit", case=False)]

* Let's see how many transit projects are in this dataset.
* <b>Tip</b>
    * The data we typically work with tends to be wide (read about wide vs. long data [here](https://www.statology.org/long-vs-wide-data/)). Scrolling horizontally gets tiresome.
    * Placing all the columns you want to temporarily work within a `list` like `preview_subset` below is a good idea to temporarily narrow down your dataframe while working. 

In [9]:
preview_subset = ["project_name", "scope_of_work"]

In [10]:
project_scores_only_transit[preview_subset]

,project_name,scope_of_work
11,Greenway Gables Managed Lanes,"managed lanes prioritizing carpools, clean vehicles, and public transit, featuring real time traffic updates and incentives for sustainable transportation choices."
16,Sparkle City Smart Streets Initiative,"an intelligent transportation system integrating traffic management, real time transit information, and smart parking solutions to enhance mobility and reduce congestion."
19,Rolling Renaissance Rabbit Express,"new, eco friendly rolling stock for public transit, incorporating advanced propulsion systems, comfortable seating, and onboard amenities."
20,Transit Treasure Transit Oasis,"transit supportive features, including shelters, wi fi, and real time information displays, prioritizing passenger convenience and accessibility."
25,Trail of Treats and Transit Hub,"a multi use path connecting to public transit, featuring public art installations, wayfinding signage, and amenities like bike storage and repair stations."
27,Park and Ride Petal Paradise,"an attractive park and ride facility with amenities like ev charging, wi fi, and convenient access to nearby transit options."
43,Brookside Bus Blossom Lane,"prioritize public transportation and enhance air quality by dedicating lanes to buses and hovs on brookside boulevard, integrating smart traffic signals and real time transit information inspired by the ancient elves."


#### Step 2: Filtering
* We've found all the projects that says "transit" somewhere in its description. 
* Now there are just many more transit related elements to go. We forgot about bikes, bus, rail, so on and so forth.
* The method above leaves us with multiple dataframes. We actually just want our one original dataframe tagged with categories. 
* A faster way: join all the keywords you want into one large string.
    * | designates "or".
    * You can read `transit_keywords` as "I want projects that contain the word transit or passenger rail or bus or ferry"

In [11]:
transit_keywords = f"({'|'.join(transit)})"

In [12]:
# Print it out
transit_keywords

'(transit|passenger rail|bus|ferry|brt)'

* Filter again - notice the .loc after df and how there are brackets around `df`?


In [13]:
project_scores_all_transit = project_scores.loc[project_scores.scope_of_work.str.contains(transit_keywords)][preview_subset]

/tmp/ipykernel_1836/3478213724.py:1: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  project_scores_all_transit = project_scores.loc[project_scores.scope_of_work.str.contains(transit_keywords)][preview_subset]


* Count how many more projects appear when we filter for 3 additional transit related keywords, compared to only transit below.

In [14]:
(
    project_scores_all_transit["project_name"].nunique()
    - project_scores_only_transit["project_name"].nunique()
)

2


* Let's put this all together. 
* I want any project that contains a transit component to be tagged as "Y" in a column called  "Transit". 
* If a project doesn't have a transit component, it gets tagged as a "N".

In [15]:
project_scores["Transit"] = np.where(
    (project_scores.scope_of_work.str.contains(transit_keywords)),
    "Y",
    "N",
)

/tmp/ipykernel_1836/1736805190.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  (project_scores.scope_of_work.str.contains(transit_keywords)),


* Using `value_counts()` we can see the total of transit related vs non-transit related projects.

In [16]:
project_scores["Transit"].value_counts()

N    35
Y     9
Name: Transit, dtype: int64

### Task 2: Functions 
* It looks like there are only  9 transit projects.
* We are missing the 2 other categories: ATP and General Lane related projects.
* We could repeat the steps above or we can use a **function.**
    * You can think of a function as a piece of code you write only once but reuse more than once.
    * In the long run, functions save you work and look neater when you present your work.
* You may not have realized this but you've been using functions this whole time.
    * When you are taking the `len()` you are using a built-in function to find the number of rows in a dataframe.

In [17]:
len(project_scores)

44

* `type` too is a built-in function that tells you what type of variable you are looking at. 

In [18]:
type(project_scores)

pandas.core.frame.DataFrame

In [19]:
type(GCS_FILE_PATH)

str

In [20]:
type(transit)

list

### Practice with outside resources
* Functions are incredibly important as such, **please spend more time than usual on this section and practice the tutorials linked.**
* [Tutorial #1 Practical Python for Data Science.](https://www.practicalpythonfordatascience.com/00_python_crash_course_functions)
* [DDS Functions.](https://docs.calitp.org/data-infra/analytics_new_analysts/01-data-analysis-intro.html#functions)

In [21]:
# Practice here
project_scores["project_cost"].map(lambda cost: int(str(cost) + "10") % 3).head()

0    1
1    2
2    1
3    2
4    2
Name: project_cost, dtype: int64

In [22]:
def if_else_function(x):
    if x > 50:
        return False
    else:
        return True
project_scores["project_cost"].map(if_else_function).head().value_counts()

False    5
Name: project_cost, dtype: int64

In [23]:
def apply_function(row):
    return 0 if if_else_function(row["project_cost"]) else row["project_cost"] % 2
project_scores.apply(apply_function, axis=1).head()

0    0
1    0
2    0
3    0
4    0
dtype: int64

####  Let's build a function together.
* This will be repetitive after the tutorials, but you will use functions all the time at DDS.
##### Step 1
* Start your function with `def` and the name you'd like. I'm calling it `categorize():`

In [24]:
def categorize():
    pass

##### Step 2 
* Now let's think of what are the two elements that we will repeat.
* We merely want to substitute `transit_keywords` with ATP or General Lane related keywords.
* Instead of the `df["Transit]"`, we want to create two new columns called something like `df["ATP]"` and `df["General_Lanes]"` to hold our yes/no results.
* Add the two elements that need to be substituted into the argument of your function.
    * It's good practice to specify what exactly the parameter should be: a string/list/dataframe/etc. 
    * Including this detail make it easier for your coworkers to read and use your code.

In [25]:
def categorize(df:pd.DataFrame, keywords:list, new_column:str):
    pass

##### Step 3
* It's also good to document what your function will return.
* In our case, it's a Pandas dataframe. 

In [26]:
def categorize(df:pd.DataFrame, keywords:list, new_column:str)->pd.DataFrame:
    pass

##### Step 4
* Think about the steps we took to categorize transit only.
* Add the sections of the code we will be reusing and sub in the original variables for the arguments.
    *  First, we joined the keywords from a list into a big string.
    *  Second, we searched through the Scope of Work column for the keywords.
    *  Third, if we find the keyword, we will tag the project as "Y" in the column "new_column". If the keyword isn't found, the project is tagged as "N".


In [27]:
def categorize(df: pd.DataFrame, keywords: list, new_column: str) -> pd.DataFrame:
    
    # Remember this used to be the list called transit_keywords, but it must be changed into a long string
    joined_keywords = f"({'|'.join(keywords)})" 

    # We are now creating a new column: notice how parameters has no quotation marks.
    df[new_column] = np.where((df.scope_of_work.str.contains(joined_keywords, regex=True)), 
        "Y",
        "N",
    )

    # We are returning the updated dataframe from this function
    return df

#### Step 5 
* Now let's use your function: input the arguments in for each of the lists that hold the categorical keywords.

In [28]:
project_scores = categorize(df = project_scores, 
                keywords = atp, 
                new_column = "ATP")

project_scores = categorize(
    df=project_scores,
    keywords=general_lanes,
    new_column="General_Lanes"
)

/tmp/ipykernel_1836/2978941839.py:7: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[new_column] = np.where((df.scope_of_work.str.contains(joined_keywords, regex=True)),


#### Check out your results
* Use the `groupby` technique from Exercise 2 to get some descriptive statistics for these 3 new columns
* Use `.reset_index()` after `aggregate()` to see what happens.
* Try `.reset_index(drop = True)` as well. 

In [29]:
project_scores.groupby("ATP")["overall_score"].agg("median").reset_index()

,ATP,overall_score
0,N,73.00
1,Y,75.00


## Function + If-Else
* There are many cases in which we want to categorize our columns to create broader groups for summarizing and aggregating.
* Using a function with an If-Else clause will help us accomplish this goal.
* **<b>Resources</b>:**
    * [DDS Apply Docs](https://docs.calitp.org/data-infra/analytics_new_analysts/01-data-analysis-intro.html#functions)
    * [DDS If-Else Tutorial](https://docs.calitp.org/data-infra/analytics_new_analysts/01-data-analysis-intro.html#if-else-statements)
    
    

In [30]:
# Practice here.

### Practice #1: 
* We are going to write an If-Else function that categorizes projects by whether it scored low, medium, or high based on its `overall_score` and percentiles.
    * If a project scores below the 25% percentile, it is a "low scoring project". If a project scores above the 25% percentile but below the 75% percentile, it is a "medium scoring project". Anything above the 75% percentile is "high scoring".
* In Data Science, we like to save our work into variables.
    * If new projects are added, then different percentiles will likely switch.
    * As such, you can save whatever percentile you like using `p75 = df.overall_score.quantile(0.75).astype(float)` which will change automatically when you load in the new data.
* Write an if-else and set the various percentiles using variables. 

In [31]:
LOW_SCORING = "low_scoring"
MEDIUM_SCORING = "medium_scoring"
HIGH_SCORING = "high_scoring"
def score_categorization(scores, low_percentile, high_percentile):
    if scores <= low_percentile:
        return LOW_SCORING
    elif scores <= high_percentile:
        return MEDIUM_SCORING
    else:
        return HIGH_SCORING

    low_percentile_value = scores.quantile(low_percentile)
    high_percentile_value = scores.quantile(high_percentile)
    scores_copy = scores.copy()
    scores_copy.loc[scores_copy <= low_percentile_value] = LOW_SCORING
    scores_copy.loc[(scores_copy <= high_percentile_value) & (scores_copy != LOW_SCORING)] = HIGH_SCORING
    scores_copy.loc[~scores_copy.isin([LOW_SCORING, HIGH_SCORING])]
    return scores_copy

score_categories = project_scores["overall_score"].map(
    lambda score: score_categorization(
        score,
        project_scores["overall_score"].quantile(0.25),
        project_scores["overall_score"].quantile(0.75)
    )
)
score_categories.value_counts() 

medium_scoring    23
low_scoring       11
high_scoring      10
Name: overall_score, dtype: int64

In [32]:
pd.concat([project_scores["overall_score"], score_categories], axis=1).head()

,overall_score,overall_score
0,72,medium_scoring
1,68,medium_scoring
2,87,high_scoring
3,75,medium_scoring
4,72,medium_scoring


### Practice #2
* Goal:
    * Above, we can see all types of combinations of categories a project can fall into. 
    * Let's do away with these "Y" and "N" columns and create actual categories in an actual column called `categories`.
    * If a project has "N" for all 3 of the General Lane, Transit, and ATP columns, it should be `Other`. 
    * If a project has "Y" for all 3, it should be categorized as "General Lane, Transit, and ATP".
    * If a project has "Y" for only ATP and Transit, it should be categorized as "Transit and ATP".
    * Yes this will be very tedious given all the combinations!
* Resource:
    * [Geeks for Geeks: if-else with multiple conditions](https://www.geeksforgeeks.org/check-multiple-conditions-in-if-statement-python/)

In [33]:
import string
OTHER = "Other"

def concatenate_categories(df_categories: pd.DataFrame) -> pd.Series:
    """
    Concatenate row-wise all values in df_categories that are equal to Y as a syntactically correct list of their column names with oxford commas
    If all values in a row are "N", replace the value with "Other"

    params:
    df_categories: a DataFrame consisting entirely of strings with values 'Y' or 'N'

    output:
    a Series of strings, with name 'combined'
    """
    categories_replaced = df_categories.copy()

    # Get the column names as a comma separated list
    for column_name in categories_replaced.columns:
        categories_replaced[column_name].replace({
            "Y": f"{column_name.replace('_', ' ')}, ",
            "N": ""
        }, inplace=True)
    concat_strings = categories_replaced.apply(
        lambda row: "".join(row.values),
        axis=1
    ).str.slice(stop=-2) # Remove trailing comma from last entry

    def add_and(s: str) -> str:
        """Replace empty strings with OTHER, otherwise replace the last ', ' with ', and' or 'and' where correct"""
        last_pos = s.rfind(", ")
        if last_pos == -1:
            return OTHER if s == "" else s
        count_commas = s.count(", ")
        if count_commas > 1:
            return f"{s[:last_pos]}, and {s[last_pos+2:]}"
        else: 
            return f"{s[:last_pos]} and {s[last_pos+2:]}"

    return concat_strings.map(add_and).rename("combined")

project_scores["category"] = concatenate_categories(
    project_scores[["Transit", "ATP", "General_Lanes"]]
)

### Please export your output as a `.parquet` to GCS before moving onto the next step

In [34]:
PROJECT_SCORES_CATEGORIZED_NAME = "anna_project_scores_categorized.parquet"
project_scores.to_parquet(f"{GCS_FILE_PATH}{PROJECT_SCORES_BY_DISTRICT_NAME}")

## For Loops 
* For Loops are one of the greatest gifts of Python. 
* It runs code from the beginning to the end of a list. 
* Below is a simple for loop that prints out all the numbers in range of 10.


In [35]:
for i in range(10):
    print(i)

0
1
2
3
4
5
6
7
8
9


* Here, I'm looping over a couple of columns in my dataframe and printing some descriptive statistics about it.
* Notice how I have to use `print` and `display` to show the results.
    * Try this same block of code without `print` and `display` to see the difference.

In [36]:
for column in ["zev_score", "vmt_score", "accessibility_score"]:
    print(f"Statistics for {column}")
    display(project_scores[column].describe())

Statistics for zev_score


count   44.00
mean     6.00
std      2.96
min      1.00
25%      3.75
50%      6.50
75%      8.00
max     10.00
Name: zev_score, dtype: float64

Statistics for vmt_score


count   44.00
mean     4.52
std      2.73
min      1.00
25%      2.00
50%      4.00
75%      6.00
max     10.00
Name: vmt_score, dtype: float64

Statistics for accessibility_score


count   44.00
mean     5.14
std      2.66
min      1.00
25%      3.00
50%      5.00
75%      7.00
max     10.00
Name: accessibility_score, dtype: float64

### Practice using a for loop
* I have aggregated the dataframe for you.

In [37]:
agg1 = (
    project_scores.groupby(["category"])
    .aggregate(
        {"overall_score": "median", "project_cost": "median", "project_name": "nunique"}
    )
    .reset_index()
    .rename(
        columns={
            "overall_score": "median_score",
            "project_cost": "median_project_cost",
            "project_name": "total_projects",
        }
    )
)

In [38]:
agg1

,category,median_score,median_project_cost,total_projects
0,ATP,72.00,4991255.00,11
1,ATP and General Lanes,82.00,5672550.50,2
2,General Lanes,73.00,5796477.00,9
3,Other,73.00,3708858.00,13
4,Transit,69.50,4399886.00,6
5,Transit and ATP,75.00,2069143.00,1
6,Transit and General Lanes,81.50,3920053.00,2


* I have also prepared an Altair chart function. 

In [39]:
def create_chart(df: pd.DataFrame, column: str) -> alt.Chart:
    title = column.replace("_", " ").title()
    chart = (
        alt.Chart(df, title=f"{title} by Categories")
        .mark_bar(size=20)
        .encode(
            x=alt.X(column),
            y=alt.Y("category"),
            color=alt.Color(
                "category",
                scale=alt.Scale(
                    range=calitp_color_palette.CALITP_CATEGORY_BRIGHT_COLORS
                ),
            ),
            tooltip=list(df.columns),
        )
        .properties(width=400, height=250)
    )
    return chart

* Use the function above to create a chart out of the aggregated dataset.

In [41]:
for column_name in agg1.columns.drop("category"):
    display(create_chart(agg1, column_name))

alt.Chart(...)

alt.Chart(...)

alt.Chart(...)


* We have a couple of other columns left that still need to be visualized. 
* This is the perfect case for using a for loop, since all we want to do is replace the column above with the two remaining columns. 
* Try this below! 
    * You'll have to create a `list` that contains the rest of the columns.
    * You'll have to wrap the function with `display()` to get your results.

## GitHub - Pull Requests
* In Exercise 1, you created a new branch that you are working on now.
* Now that you are done with Exercise 3, you are at a nice stopping point to commit your work to our `main` branch.

**Steps**
1. Do the normal workflow of `committing` your work. 
2. Navigate to the our `data-analyses` [repo over here](https://github.com/cal-itp/data-analyses).
3. Follow the steps detailed in [this video](https://youtu.be/nCKdihvneS0?si=nPlBOAMcgO1nv3v1&t=95). 
4. Once you're done writing, scroll down the bottom and click `merge pull request` 
<img src= "./starter_kit_img.png">
5. Your work is now merged into the `main` branch of our `data-analyses` repo. 
6. To check, navigate to the our [repo](https://github.com/cal-itp/data-analyses) and to this `starter_kit` folder to make sure your notebooks are on the `main` branch.
7. Delete the branch `your_branch`. 
    * It's considered outdated now because your changes are on the `main branch`. In the terminal, paste `git branch -d your_branch`. 
    * If that doesn't work, paste `git branch -D your_branch`.
8. Continuing in the terminal, paste `git switch main`. 
9. Paste `git pull origin main`. 
    * This pulls down the work you just uploaded, along with the other work your coworkers have committed onto the main branch. 
9. Create a new branch `git switch -c your_branch` to continue working on exercises 4 and 5.
    * Your new branch can have the same name as the branch you just merged in.